In [1]:
%matplotlib inline
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# ── Paths & mappings ────────────────────────────────────────────────────────
RESULTS_DIR = Path("../data/test_results")

MODEL_META = {
    "dcrnn_wind_dcrnn_base":     ("DCRNN",   "BASE"),
    "dcrnn_wind_dcrnn":          ("DCRNN",   "GRID"),
    "dcrnn_wind_dcrnn_nwp_hist": ("DCRNN",   "GRID+HIST"),
    "mtgnn_wind_mtgnn":          ("MTGNN",   "BASE"),
    "mtgnn_wind_mtgnn_nwp":      ("MTGNN",   "GRID"),
    "mtgnn_wind_mtgnn_nwp_hist": ("MTGNN",   "GRID+HIST"),
    "wavenet_wind_wavenet":          ("WaveNet", "BASE"),
    "wavenet_wind_wavenet_nwp":      ("WaveNet", "GRID"),
    "wavenet_wind_wavenet_nwp_hist": ("WaveNet", "GRID+HIST"),
    # NWP baselines (from evaluate_reference.py)
    "icon_d2":                   ("ICON-D2", "REF"),
    "ecmwf":                     ("ECMWF",   "REF"),
}

# Fold-Zeiträume sind SPLIT-abhängig — Val- und Test-Sets nutzen andere Fenster.
FOLD_LABELS_VAL  = {0: "Aug–Nov 2024", 1: "Nov 2024–Mar 2025", 2: "Apr–Aug 2025"}
FOLD_LABELS_TEST = {0: "Aug–Nov 2025", 1: "Dez 2025–März 2026"}
FOLD_LABELS_BY_SPLIT = {"val": FOLD_LABELS_VAL, "test": FOLD_LABELS_TEST}

# Backward-compat: FOLD_LABELS / FOLD_LABEL_TO_IDX referenzieren weiterhin Val.
FOLD_LABELS       = FOLD_LABELS_VAL
FOLD_LABEL_TO_IDX = {v: k for k, v in FOLD_LABELS_VAL.items()}

def _fold_labels_for(split):
    return FOLD_LABELS_BY_SPLIT.get(str(split).lower(), FOLD_LABELS_VAL)

def _fold_idx_for(split, label):
    """fold_label (z.B. 'Aug–Nov 2025') → fold-Index, split-abhängig."""
    return {v: k for k, v in _fold_labels_for(split).items()}.get(label)

# Eval scripts produce: station_id, mae, rmse, r2, skill, skill_nwp, n_samples
# Notebook expects:     station,    val_mae, val_rmse, val_r2, skill_icond2, skill_ecmwf
_COL_RENAME = {
    "station_id": "station",
    "mae":        "val_mae",
    "rmse":       "val_rmse",
    "r2":         "val_r2",
    "skill_nwp":  "skill_icond2",
    "skill":      "skill_pers",
}

# ── Model CSV loader ────────────────────────────────────────────────────────
def load_all_csvs():
    frames = []
    for path in sorted(RESULTS_DIR.glob("*.csv")):
        m = re.match(r"(.+)_fold(\d+)$", path.stem)
        if not m:
            continue
        prefix, fold = m.group(1), int(m.group(2))
        # Test-Outputs heißen <prefix>_test_fold{n} → split erkennen + Suffix strippen.
        split = "val"
        if prefix.endswith("_test"):
            split, prefix = "test", prefix[:-len("_test")]
        if prefix not in MODEL_META:
            continue
        model_name, variant = MODEL_META[prefix]
        df = pd.read_csv(path).rename(columns=_COL_RENAME)
        if "scenario"    not in df.columns: df["scenario"]    = "excl_val"
        if "seen"        not in df.columns: df["seen"]        = "n"
        if "skill_ecmwf" not in df.columns: df["skill_ecmwf"] = np.nan
        df["model"]      = model_name
        df["variant"]    = variant
        df["fold"]       = fold
        df["split"]      = split
        df["fold_label"] = _fold_labels_for(split).get(fold, "?")
        frames.append(df)
    if not frames:
        print("No CSV files found in", RESULTS_DIR)
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

# ── Reference metrics loader (legacy — superseded by evaluate_reference.py parquets) ────
def load_reference_metrics():
    path = RESULTS_DIR / "reference_metrics.csv"
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    df["fold_label"] = df["fold"].map(FOLD_LABELS)
    print(f"[reference_metrics.csv] {len(df):,} rows")
    return df

data    = load_all_csvs()
ref_all = load_reference_metrics()

print(f"Model data: {len(data):,} rows | "
      f"{data['model'].nunique() if not data.empty else 0} models | "
      f"splits: {sorted(data['split'].unique()) if not data.empty else []}")
if not data.empty:
    print("Models:", sorted(data["model"].unique()))
    for sp in sorted(data["split"].unique()):
        folds = sorted(data[data["split"] == sp]["fold"].unique())
        print(f"  split='{sp}': folds {folds}")


In [ ]:
# ── Shared helpers ───────────────────────────────────────────────────────────

def ref_rmse_from_skill(df, ref_col):
    """Fallback: derive per-station reference RMSE from skill score."""
    denom = (1.0 - df[ref_col]).clip(lower=0.001, upper=0.999)
    return df["val_rmse"] / denom


def filter_model(df, variant, fold, scenario, seen, split="Val"):
    # Split zuerst (Val/Test haben getrennte Stationen UND Fold-Zeiträume).
    if "split" in df.columns:
        df = df[df["split"] == str(split).lower()]
    # WaveNet hat inzwischen echte BASE/GRID/GRID+HIST-Varianten (siehe MODEL_META) —
    # einfacher Variant-Filter für alle Modelle, kein Spezialfall mehr nötig.
    df = df[df["variant"] == variant].copy()
    if fold != "Average":
        df = df[df["fold"] == _fold_idx_for(split, fold)]
    df = df[df["scenario"] == scenario]
    if seen == "Val stations only":
        df = df[df["seen"] == "n"]
    elif seen == "Train stations only":
        df = df[df["seen"] == "y"]
    return df


def filter_ref(df_ref, fold, seen, period="val", horizon=-1):
    """Filter reference_metrics to the aggregate val period for one fold."""
    if df_ref.empty:
        return df_ref
    df = df_ref[(df_ref["period"] == period) & (df_ref["horizon"] == horizon)].copy()
    if fold != "Average":
        df = df[df["fold"] == FOLD_LABEL_TO_IDX[fold]]
    if seen == "Val stations only":
        df = df[df["seen"] == "n"]
    elif seen == "Train stations only":
        df = df[df["seen"] == "y"]
    return df


def _bar_stats(series):
    return float(np.nanmean(series)), float(np.nanstd(series))


# ── Split + Fold Widgets (gekoppelt) ─────────────────────────────────────────
# Beim Umschalten des Splits aktualisiert ein observe()-Callback die Fold-Optionen,
# weil Val (3 Folds) und Test (1 Fold) andere Zeiträume haben.
def _available_splits():
    if not data.empty and "split" in data.columns:
        present = set(data["split"].unique())
        return [s.capitalize() for s in ["val", "test"] if s in present] or ["Val"]
    return ["Val"]

def make_split_fold(style, layout, fold_desc="Fold:"):
    splits = _available_splits()
    w_split = widgets.Dropdown(options=splits, value=splits[0],
                               description="Split:", style=style, layout=layout)
    w_fold  = widgets.Dropdown(
        options=list(_fold_labels_for(splits[0]).values()) + ["Average"],
        value="Average", description=fold_desc, style=style, layout=layout)
    def _on_split(change):
        w_fold.options = list(_fold_labels_for(change["new"]).values()) + ["Average"]
        w_fold.value   = "Average"
    w_split.observe(_on_split, names="value")
    return w_split, w_fold


---
## Plot 1 — Model Comparison (Bar Chart)

In [4]:
def plot_bar(metric, variant, fold, scenario, seen, split):
    df = filter_model(data, variant, fold, scenario, seen, split)

    if df.empty:
        print("No data available for the current selection.")
        return

    MODELS = ["DCRNN", "MTGNN", "WaveNet"]
    bars, vals, errs, colors = [], [], [], []

    def _add_model(col):
        for m in MODELS:
            s = df[df["model"] == m][col].dropna()
            if s.empty: continue
            bars.append(m); v, e = _bar_stats(s); vals.append(v); errs.append(e)
            colors.append("steelblue")

    def _add_ref_from_data(col):
        """ICON-D2 / ECMWF bars from evaluate_reference.py CSVs (loaded into data)."""
        ref_df = data[data["model"].isin(["ICON-D2", "ECMWF"]) & (data["split"] == split.lower())]
        if fold != "Average":
            ref_df = ref_df[ref_df["fold"] == _fold_idx_for(split, fold)]
        for ref_model, color in [("ICON-D2", "#888888"), ("ECMWF", "#555555")]:
            s = ref_df[ref_df["model"] == ref_model][col].dropna()
            if s.empty: continue
            bars.append(ref_model); v, e = _bar_stats(s); vals.append(v); errs.append(e)
            colors.append(color)

    if metric == "RMSE":
        _add_model("val_rmse");          _add_ref_from_data("val_rmse"); ylabel = "RMSE (m/s)"
    elif metric == "MAE":
        _add_model("val_mae");           _add_ref_from_data("val_mae");  ylabel = "MAE (m/s)"
    elif metric == "R²":
        _add_model("val_r2");            _add_ref_from_data("val_r2");   ylabel = "R²"
    elif metric in ("Skill ICON-D2", "Skill ECMWF"):
        skill_col = "skill_icond2" if "ICON-D2" in metric else "skill_ecmwf"
        _add_model(skill_col)
        bars  += ["ICON-D2", "ECMWF"]; vals  += [0.0, 0.0]
        errs  += [0.0, 0.0];           colors += ["#888888", "#555555"]
        ylabel = f"Skill vs. {'ICON-D2' if 'ICON-D2' in metric else 'ECMWF'}"

    if not bars:
        print("No data available for the current selection.")
        return

    fig, ax = plt.subplots(figsize=(max(6, len(bars) * 1.3), 5))
    x = np.arange(len(bars))
    ax.bar(x, vals, yerr=errs, capsize=5, color=colors,
           edgecolor="white", linewidth=0.5,
           error_kw={"elinewidth": 1.5, "alpha": 0.7})

    if metric == "R²":
        ymin = min((v for v in vals if np.isfinite(v)), default=-0.1) - 0.05
        ax.set_ylim(min(ymin, -0.05), 1.12)
        ax.axhline(0.0, color="red", linestyle="--", linewidth=1.5, alpha=0.8)
        ax.axhline(1.0, color="red", linestyle="--", linewidth=1.5, alpha=0.8)
    elif metric in ("Skill ICON-D2", "Skill ECMWF"):
        ax.axhline(0.0, color="gray", linestyle="--", linewidth=1.0, alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(bars, fontsize=12)
    ax.tick_params(axis="y", labelsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.axhline(0, color="gray", linewidth=0.6)

    fold_text = fold if fold != "Average" else "Average (all folds)"
    ax.text(
        0.98, 0.98,
        f"Split: {split}\nFold: {fold_text}\nScenario: {scenario}\nStations: {seen}",
        transform=ax.transAxes, ha="right", va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#ccc", alpha=0.85),
    )
    plt.tight_layout()
    plt.show(); plt.close()


_style  = {"description_width": "70px"}
_layout = widgets.Layout(width="200px")

w1_metric   = widgets.Dropdown(
    options=["RMSE", "MAE", "R²", "Skill ICON-D2", "Skill ECMWF"],
    value="RMSE", description="Metric:", style=_style, layout=_layout)
w1_variant  = widgets.Dropdown(
    options=["BASE", "GRID", "GRID+HIST"],
    value="BASE", description="Variant:", style=_style, layout=_layout)
w1_split, w1_fold = make_split_fold(_style, _layout)
w1_scenario = widgets.Dropdown(
    options=["excl_val", "incl_val"],
    value="excl_val", description="Scenario:", style=_style, layout=_layout)
w1_seen     = widgets.Dropdown(
    options=["Val stations only", "Train stations only", "Both combined"],
    value="Val stations only", description="Stations:", style=_style, layout=_layout)

out1 = widgets.interactive_output(
    plot_bar,
    {"metric": w1_metric, "variant": w1_variant, "fold": w1_fold,
     "scenario": w1_scenario, "seen": w1_seen, "split": w1_split},
)
display(widgets.HBox([w1_split, w1_metric, w1_variant, w1_fold, w1_scenario, w1_seen]), out1)

Output()

---
## Plot 2 — Station-Level Scatter (Model vs. Reference)

In [ ]:
# Metric column mapping: (model_col, xlabel_unit, higher_is_better)
_METRIC_MAP = {
    "RMSE": ("val_rmse", "RMSE (m/s)", False),
    "MAE":  ("val_mae",  "MAE (m/s)",  False),
    "R²":   ("val_r2",   "R²",         True),
}


def _get_ref_per_station(model_col, fold, split):
    """Per-station reference metrics from evaluate_reference.py CSVs (in data), split-aware."""
    result = {}
    for ref_mdl in ["ICON-D2", "ECMWF"]:
        r = data[(data["model"] == ref_mdl) & (data["split"] == split.lower())][
            ["station", "fold", model_col]].copy()
        if r.empty:
            result[ref_mdl] = pd.DataFrame()
            continue
        if fold == "Average":
            agg = r.groupby("station")[model_col].mean().reset_index()
        else:
            agg = r[r["fold"] == _fold_idx_for(split, fold)][["station", model_col]].copy()
        result[ref_mdl] = agg.rename(columns={model_col: "ref"})
    return result


def _scatter_panel(ax, x, y, metric_name, xlabel):
    higher_better = _METRIC_MAP[metric_name][2]
    better = (y > x) if higher_better else (y < x)
    worse  = ~better

    ax.scatter(x[better], y[better], c="#2ca02c", alpha=0.55, s=28, label=f"Better  (n={better.sum()})")
    ax.scatter(x[worse],  y[worse],  c="#d62728", alpha=0.55, s=28, label=f"Worse   (n={worse.sum()})")

    finite = np.concatenate([x, y])
    finite = finite[np.isfinite(finite)]
    if len(finite) == 0:
        return

    if higher_better:
        lo = min(np.nanmin(finite) - 0.05, 0.0)
        hi = max(np.nanmax(finite) + 0.05, 1.0)
        ax.plot([lo, hi], [lo, hi], "--", color="#aaa", linewidth=1.2, zorder=0)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    else:
        lim_max = max(np.nanpercentile(finite, 99) * 1.08, 0.3)
        ax.plot([0, lim_max], [0, lim_max], "--", color="#aaa", linewidth=1.2, zorder=0)
        ax.set_xlim(0, lim_max); ax.set_ylim(0, lim_max)

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(f"Model {metric_name}", fontsize=12)
    ax.tick_params(labelsize=12)
    ax.legend(fontsize=10, loc="upper left", framealpha=0.85)


def plot_scatter(metric, model, variant, fold, scenario, seen, split):
    model_col, metric_unit, _ = _METRIC_MAP[metric]

    df   = filter_model(data, variant, fold, scenario, seen, split)
    df_m = df[df["model"] == model].copy() if not df.empty else pd.DataFrame()
    if df_m.empty:
        print(f"No data for model '{model}' with the current selection.")
        return

    # Per-station model values
    if fold == "Average":
        m_agg = df_m.groupby("station")[model_col].mean().reset_index()
    else:
        m_agg = df_m[df_m["fold"] == _fold_idx_for(split, fold)][["station", model_col]].copy()

    # Per-station reference values from evaluate_reference.py CSVs
    refs = _get_ref_per_station(model_col, fold, split)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    for ax, ref_name in [(ax1, "ICON-D2"), (ax2, "ECMWF")]:
        ref_df = refs.get(ref_name, pd.DataFrame())
        ax.set_title(f"Model vs {ref_name}", fontsize=12)

        if ref_df.empty:
            ax.text(0.5, 0.5, f"No {ref_name} reference data\n(run evaluate_reference.py)",
                    ha="center", va="center", transform=ax.transAxes, fontsize=11)
            continue

        merged = m_agg.merge(ref_df, on="station", how="inner").dropna(subset=[model_col, "ref"])
        if len(merged) < 2:
            ax.text(0.5, 0.5, f"Insufficient data for {ref_name}",
                    ha="center", va="center", transform=ax.transAxes, fontsize=11)
            continue

        x  = merged["ref"].values
        y  = merged[model_col].values
        ok = np.isfinite(x) & np.isfinite(y)
        if ok.sum() < 2:
            ax.text(0.5, 0.5, "No finite pairs", ha="center", va="center",
                    transform=ax.transAxes, fontsize=11)
            continue

        _scatter_panel(ax, x[ok], y[ok], metric, f"{ref_name} {metric_unit}")

    fold_text = fold if fold != "Average" else "Average (all folds)"
    fig.text(0.5, -0.02,
             f"Split: {split} | Model: {model} | Variant: {variant} | Metric: {metric} | "
             f"Fold: {fold_text} | Scenario: {scenario} | Stations: {seen}",
             ha="center", fontsize=10, color="#555")
    plt.tight_layout()
    plt.show(); plt.close()


_available_models = [m for m in ["DCRNN", "MTGNN", "WaveNet"]
                     if not data.empty and m in data["model"].values] or ["DCRNN", "MTGNN", "WaveNet"]

w2_metric   = widgets.Dropdown(options=["RMSE", "MAE", "R²"],
                               value="RMSE", description="Metric:", style=_style, layout=_layout)
w2_model    = widgets.Dropdown(options=_available_models, value=_available_models[0],
                               description="Model:", style=_style, layout=_layout)
w2_variant  = widgets.Dropdown(options=["BASE", "GRID", "GRID+HIST"],
                               value="BASE", description="Variant:", style=_style, layout=_layout)
w2_split, w2_fold = make_split_fold(_style, _layout)
w2_scenario = widgets.Dropdown(options=["excl_val", "incl_val"],
                               value="excl_val", description="Scenario:", style=_style, layout=_layout)
w2_seen     = widgets.Dropdown(options=["Val stations only", "Train stations only", "Both combined"],
                               value="Val stations only", description="Stations:", style=_style, layout=_layout)

out2 = widgets.interactive_output(
    plot_scatter,
    {"metric": w2_metric, "model": w2_model, "variant": w2_variant,
     "fold": w2_fold, "scenario": w2_scenario, "seen": w2_seen, "split": w2_split},
)
display(widgets.HBox([w2_split, w2_metric, w2_model, w2_variant, w2_fold, w2_scenario, w2_seen]), out2)

Output()

In [ ]:
# ── Plot 3: Summary table ────────────────────────────────────────────────────
_SEEN_TO_FILTER = {
    "Train": "Train stations only",
    "Val":   "Val stations only",
    "Both":  "Both combined",
}
_SEEN_TO_DATA = {"Train": "y", "Val": "n", "Both": None}
_SUMMARY_MODELS = ["DCRNN", "MTGNN", "WaveNet"]


def show_summary_table(variant, seen, split, fold):
    seen_val = _SEEN_TO_DATA[seen]
    sp = split.lower()

    mask = (data["split"] == sp) & data["model"].isin(_SUMMARY_MODELS) & (data["variant"] == variant)
    if fold != "Average":
        mask &= data["fold"] == _fold_idx_for(split, fold)
    if seen_val is not None:
        mask &= data["seen"] == seen_val
    df_filt = data[mask]

    agg = (
        df_filt.groupby("model")[["val_r2", "val_rmse", "skill_icond2", "skill_ecmwf"]]
        .mean()
        .reindex(_SUMMARY_MODELS)
        .rename(columns={
            "val_r2":        "R²",
            "val_rmse":      "RMSE",
            "skill_icond2":  "Skill ICON-D2",
            "skill_ecmwf":   "Skill ECMWF",
        })
    )

    # Reference R² and RMSE from evaluate_reference.py CSVs (in data), split-aware
    for ref_mdl, label in [("ICON-D2", "ICON-D2"), ("ECMWF", "ECMWF")]:
        ref_sub = data[(data["model"] == ref_mdl) & (data["split"] == sp)]
        if fold != "Average":
            ref_sub = ref_sub[ref_sub["fold"] == _fold_idx_for(split, fold)]
        if not ref_sub.empty:
            agg[f"R² {label}"]   = ref_sub["val_r2"].mean()
            agg[f"RMSE {label}"] = ref_sub["val_rmse"].mean()

    grad_up   = [c for c in ["R²", "Skill ICON-D2", "Skill ECMWF",
                              "R² ICON-D2", "R² ECMWF"] if c in agg.columns]
    grad_down = [c for c in ["RMSE", "RMSE ICON-D2", "RMSE ECMWF"] if c in agg.columns]

    fold_text = fold if fold != "Average" else "Ø alle Folds"
    styled = (
        agg.style
        .format("{:.4f}", na_rep="—")
        .background_gradient(subset=grad_up,   cmap="RdYlGn",   axis=0)
        .background_gradient(subset=grad_down, cmap="RdYlGn_r", axis=0)
        .set_caption(f"Split: {split} | Fold: {fold_text} | Variant: {variant} | Seen: {seen} | Ø über Stationen")
    )
    display(styled)


w3_split, w3_fold = make_split_fold(_style, _layout)
w3_variant = widgets.Dropdown(
    options=["BASE", "GRID", "GRID+HIST"],
    value="BASE", description="Variant:", style=_style, layout=_layout)
w3_seen = widgets.Dropdown(
    options=["Train", "Val", "Both"],
    value="Val", description="Seen:", style=_style, layout=_layout)

out3 = widgets.interactive_output(
    show_summary_table, {"variant": w3_variant, "seen": w3_seen, "split": w3_split, "fold": w3_fold})
display(widgets.HBox([w3_split, w3_fold, w3_variant, w3_seen]), out3)


In [ ]:
# ── Plot 3b: Comparison bar chart — ICON-D2 vs NWP only vs NWP + Hist ───────
def _improvement_bracket(ax, x0, x1, v0, v1, higher_better, lift=0.0,
                          color="dimgray", fontsize=11):
    """
    Significance bracket placed in the space between bar tops and ylim.
    `lift` = 0.0 places the bracket low in that space, 1.0 places it high.
    """
    if np.isnan(v0) or np.isnan(v1):
        return
    pct  = (v1 - v0) / abs(v0) * 100 if higher_better else (v0 - v1) / abs(v0) * 100
    sign = "+" if pct >= 0 else ""

    _, ymax    = ax.get_ylim()
    headroom   = ymax - max(v0, v1)        # space above the taller bar
    tick       = headroom * 0.10
    y_top      = max(v0, v1) + headroom * (0.28 + lift * 0.40)

    ax.plot([x0, x0, x1, x1],
            [v0 + tick, y_top, y_top, v1 + tick],
            color=color, lw=1.2)
    ax.text((x0 + x1) / 2, y_top + headroom * 0.05,
            f"{sign}{pct:.1f}%", ha="center", va="bottom",
            fontsize=fontsize, color=color, fontweight="bold")


def show_comparison_plot(seen, split, fold):
    seen_val = _SEEN_TO_DATA[seen]
    sp = split.lower()

    # --- Reference (ICON-D2) from evaluate_reference.py CSVs (loaded into `data`), split-aware ---
    def _ref_mean(col):
        m = (data["model"] == "ICON-D2") & (data["split"] == sp)
        if fold != "Average":
            m &= data["fold"] == _fold_idx_for(split, fold)
        if seen_val is not None:
            m &= data["seen"] == seen_val
        s = data[m][col]
        return float(s.mean()) if not s.empty else np.nan

    r2_ref   = _ref_mean("val_r2")
    rmse_ref = _ref_mean("val_rmse")

    # --- Model means (DCRNN GRID / MTGNN GRID+HIST, fixed) ---
    def _mean(model, variant, col):
        m = (data["model"] == model) & (data["variant"] == variant) & (data["split"] == sp)
        if fold != "Average":
            m &= data["fold"] == _fold_idx_for(split, fold)
        if seen_val is not None:
            m &= data["seen"] == seen_val
        s = data[m][col]
        return float(s.mean()) if not s.empty else np.nan

    r2_vals   = [r2_ref,
                 _mean("DCRNN", "GRID",      "val_r2"),
                 _mean("MTGNN", "GRID+HIST", "val_r2")]
    rmse_vals = [rmse_ref,
                 _mean("DCRNN", "GRID",      "val_rmse"),
                 _mean("MTGNN", "GRID+HIST", "val_rmse")]

    labels = ["ICON-D2", "NWP only", "NWP + Hist"]
    green  = "#2ca02c"
    x      = np.arange(len(labels))
    bar_w  = 0.6

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # ── Left: R² ─────────────────────────────────────────────────────────────
    ax1.bar(x, r2_vals, color=green, alpha=0.85, width=bar_w, zorder=3)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=12)
    ax1.set_ylabel("R²", fontsize=12)
    ax1.tick_params(labelsize=12)
    ax1.set_ylim(0, 1.0)
    ax1.set_yticks(np.arange(0, 1.1, 0.2))
    ax1.set_yticklabels([f"{v:.1f}" for v in np.arange(0, 1.1, 0.2)], fontsize=12)

    # Dashed container outlines (0 → 1) per bar
    for xi in x:
        rect = mpatches.Rectangle(
            (xi - bar_w / 2, 0), bar_w, 1.0,
            fill=False, linestyle="--", edgecolor="gray",
            linewidth=0.9, alpha=0.65, zorder=4,
        )
        ax1.add_patch(rect)

    # Brackets inside the headroom between bar tops and y=1
    _improvement_bracket(ax1, 0, 1, r2_vals[0], r2_vals[1], higher_better=True, lift=0.0)
    _improvement_bracket(ax1, 0, 2, r2_vals[0], r2_vals[2], higher_better=True, lift=1.0)

    ax1.set_title("R²", fontsize=12)

    # ── Right: RMSE ──────────────────────────────────────────────────────────
    ax2.bar(x, rmse_vals, color=green, alpha=0.85, width=bar_w, zorder=3)
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels, fontsize=12)
    ax2.set_ylabel("RMSE (m/s)", fontsize=12)
    ax2.tick_params(labelsize=12)

    max_rmse = max((v for v in rmse_vals if not np.isnan(v)), default=1.0)
    ax2.set_ylim(0, max_rmse * 1.45)

    _improvement_bracket(ax2, 0, 1, rmse_vals[0], rmse_vals[1], higher_better=False, lift=0.0)
    _improvement_bracket(ax2, 0, 2, rmse_vals[0], rmse_vals[2], higher_better=False, lift=1.0)

    ax2.set_title("RMSE", fontsize=12)

    fold_text = fold if fold != "Average" else "Ø alle Folds"
    fig.suptitle(f"NWP baseline comparison — Split: {split} | Fold: {fold_text} | Stations: {seen}",
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
    plt.close()


out_cmp = widgets.interactive_output(
    show_comparison_plot, {"seen": w3_seen, "split": w3_split, "fold": w3_fold})
display(out_cmp)


In [8]:
# ── Raw predictions loader ────────────────────────────────────────────────────
RAW_DIR = Path("../data/raw_preds")

def load_raw_preds():
    frames = []
    for path in sorted(RAW_DIR.glob("*_raw.parquet")):
        # Filename: {prefix}_fold{n}_raw.parquet  (Test: {prefix}_test_fold{n}_raw.parquet)
        stem = path.stem[:-4]  # strip _raw suffix (4 chars)
        m = re.match(r"(.+)_fold(\d+)$", stem)
        if not m:
            continue
        prefix = m.group(1)
        fold   = int(m.group(2))
        split  = "val"
        if prefix.endswith("_test"):
            split, prefix = "test", prefix[:-len("_test")]
        if prefix not in MODEL_META:
            continue
        model_name, variant = MODEL_META[prefix]
        df = pd.read_parquet(path)
        df["model"]      = model_name
        df["variant"]    = variant
        df["fold"]       = fold
        df["split"]      = split
        df["fold_label"] = _fold_labels_for(split).get(fold, "?")
        df["hour"]       = pd.to_datetime(df["valid_time"]).dt.hour
        df["month"]      = pd.to_datetime(df["valid_time"]).dt.month
        # run_hour = Uhrzeit, zu der die Prognose erstellt wurde (ICON-D2-Lauf: 6/9/12/15)
        df["run_hour"]   = pd.to_datetime(df["run_time"]).dt.hour
        frames.append(df)
    if not frames:
        print(f"Keine Rohdaten in {RAW_DIR} — bitte get_test_results_*.py ausführen.")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

raw = load_raw_preds()
print(f"Raw predictions: {len(raw):,} Zeilen | {raw['model'].nunique() if not raw.empty else 0} Modelle")
if not raw.empty:
    print("Splits:", sorted(raw["split"].unique()),
          "| Prognose-Laufzeiten (run_hour):", sorted(raw["run_hour"].unique()))

Raw predictions: 36,566,400 Zeilen | 4 Modelle
Splits: ['test', 'val'] | Prognose-Laufzeiten (run_hour): [6, 9, 12, 15]


---
## Plot 4 — Fehler nach Prognosehorizont

In [9]:
# ── Shared helper for raw stratification ─────────────────────────────────────
_MODEL_COLORS = {
    "DCRNN":   "steelblue",
    "MTGNN":   "darkorange",
    "WaveNet": "forestgreen",
    "ICON-D2": "#888888",
    "ECMWF":   "#555555",
}

def _metric_from_err(pred, gt, metric):
    err = pred - gt
    if metric == "RMSE":
        return float(np.sqrt((err ** 2).mean()))
    elif metric == "MAE":
        return float(err.abs().mean())
    elif metric == "R²":
        ss_res = float((err ** 2).sum())
        ss_tot = float(((gt - gt.mean()) ** 2).sum())
        return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    raise ValueError(f"Unknown metric {metric}")

def _compute_metric(grp, metric):
    return _metric_from_err(grp["pred"], grp["gt"], metric)

def _compute_metric_ref(grp, metric, col):
    return _metric_from_err(grp[col], grp["gt"], metric)

def _metric_ylabel(metric):
    return metric if metric == "R²" else f"{metric} (m/s)"


def plot_horizon(metric, variant, fold, run_hour, split):
    if raw.empty:
        print("Keine Rohdaten vorhanden.")
        return

    df = raw[raw["split"] == split.lower()].copy()
    if variant != "All":
        df = df[df["variant"] == variant]
    if fold != "Average":
        df = df[df["fold"] == _fold_idx_for(split, fold)]
    if run_hour != "Alle":
        df = df[df["run_hour"] == int(run_hour)]
    if df.empty:
        print("Keine Daten für diese Auswahl.")
        return

    fig, ax = plt.subplots(figsize=(12, 5))

    # Trained models
    for mdl in ["DCRNN", "MTGNN", "WaveNet"]:
        dm = df[df["model"] == mdl]
        if dm.empty:
            continue
        vals = dm.groupby("horizon").apply(lambda g: _compute_metric(g, metric))
        ax.plot(vals.index, vals.values, label=mdl,
                color=_MODEL_COLORS.get(mdl), linewidth=2, marker=".", markersize=4)

    # NWP baselines from raw parquets (icon_d2 / ecmwf via MODEL_META)
    for ref_model in ["ICON-D2", "ECMWF"]:
        dm = df[df["model"] == ref_model]
        if not dm.empty:
            vals = dm.groupby("horizon").apply(lambda g: _compute_metric(g, metric))
            ax.plot(vals.index, vals.values, label=ref_model,
                    color=_MODEL_COLORS.get(ref_model), linewidth=1.5, linestyle="--")

    # Fallback: ICON-D2 from nwp_ref column (if NWP baselines not in raw)
    ref_models_present = {"ICON-D2", "ECMWF"} & set(df["model"].unique())
    if not ref_models_present and "nwp_ref" in df.columns:
        nwp_vals = df.groupby("horizon").apply(lambda g: _compute_metric_ref(g, metric, "nwp_ref"))
        ax.plot(nwp_vals.index, nwp_vals.values, label="ICON-D2 (nwp_ref)",
                color="#888", linewidth=1.5, linestyle="--")

    ax.set_xlabel("Prognosehorizont (Stunden)", fontsize=12)
    ax.set_ylabel(_metric_ylabel(metric), fontsize=12)
    ax.legend(fontsize=11, loc="upper left")
    ax.grid(axis="y", alpha=0.4)
    fold_text = fold if fold != "Average" else "Ø alle Folds"
    run_text  = "alle Läufe" if run_hour == "Alle" else f"{run_hour}-Uhr-Lauf"
    ax.set_title(f"{metric} nach Prognosehorizont | Split: {split} | Variante: {variant} | "
                 f"{fold_text} | {run_text}", fontsize=13)
    plt.tight_layout()
    plt.show(); plt.close()


_style4  = {"description_width": "90px"}
_layout4 = widgets.Layout(width="210px")

_run_hour_opts = ["Alle"] + (sorted(int(h) for h in raw["run_hour"].unique()) if not raw.empty else [6, 9, 12, 15])

w4_metric   = widgets.Dropdown(options=["RMSE", "MAE", "R²"], value="RMSE",
                               description="Metrik:", style=_style4, layout=_layout4)
w4_variant  = widgets.Dropdown(options=["All", "BASE", "GRID", "GRID+HIST", "REF"], value="BASE",
                               description="Variante:", style=_style4, layout=_layout4)
w4_split, w4_fold = make_split_fold(_style4, _layout4)
w4_runhour  = widgets.Dropdown(options=_run_hour_opts, value="Alle",
                               description="Prognose-Lauf:", style=_style4, layout=_layout4)

out4 = widgets.interactive_output(
    plot_horizon, {"metric": w4_metric, "variant": w4_variant, "fold": w4_fold,
                   "run_hour": w4_runhour, "split": w4_split}
)
display(widgets.HBox([w4_split, w4_metric, w4_variant, w4_fold, w4_runhour]), out4)

Output()

---
## Plot 5 — Fehler nach Windgeschwindigkeit / Tagesstunde / Monat / Horizont (Boxplots)

In [10]:
# ── Plot 5 helper ────────────────────────────────────────────────────────────
_WS_BINS = list(range(0, 22, 2))   # [0,2,4,...,20] = 11 edges
# Intervall-Labels: linke Grenze inklusiv [, rechte exklusiv ) — right=False in pd.cut.
# 10 reguläre Intervalle + offenes Endintervall = 11 Labels (passt zu bins+[inf]).
_WS_LABELS = [f"[{_WS_BINS[i]}, {_WS_BINS[i+1]})" for i in range(len(_WS_BINS) - 1)] + ["[20, ∞)"]

_X_OPTS = {
    "Prognosehorizont": ("horizon",  "Prognosehorizont (h)"),
    "Tagesstunde":      ("hour",     "Stunde (UTC)"),
    "Monat":            ("month",    "Monat"),
    "WS-Klasse (m/s)":  ("ws_bin",  "Windgeschwindigkeit (m/s)  [links inkl., rechts exkl.)"),
}


def plot_stratified(x_axis, metric, variant, fold, plot_type, split):
    if raw.empty:
        print("Keine Rohdaten vorhanden.")
        return

    df = raw[raw["split"] == split.lower()].copy()
    if variant != "All":
        df = df[df["variant"] == variant]
    if fold != "Average":
        df = df[df["fold"] == _fold_idx_for(split, fold)]
    if df.empty:
        print("Keine Daten für diese Auswahl.")
        return

    x_col, xlabel = _X_OPTS[x_axis]

    if x_col == "ws_bin":
        df = df.copy()
        df["ws_bin"] = pd.cut(df["gt"], bins=_WS_BINS + [np.inf], labels=_WS_LABELS, right=False)
        df = df.dropna(subset=["ws_bin"])

    models = [m for m in ["DCRNN", "MTGNN", "WaveNet"] if m in df["model"].unique()]
    if not models:
        print("Keine Modell-Daten für diese Auswahl.")
        return

    if plot_type == "Linie":
        fig, ax = plt.subplots(figsize=(12, 5))
        for mdl in models:
            dm = df[df["model"] == mdl]
            vals = dm.groupby(x_col).apply(lambda g: _compute_metric(g, metric))
            ax.plot(
                range(len(vals)) if x_col == "ws_bin" else vals.index,
                vals.values, label=mdl, color=_MODEL_COLORS.get(mdl), linewidth=2, marker="."
            )
        if x_col == "ws_bin":
            ax.set_xticks(range(len(_WS_LABELS)))
            ax.set_xticklabels(_WS_LABELS, rotation=45, ha="right")
        ax.set_xlabel(xlabel, fontsize=12)
        ax.set_ylabel(f"{metric} (m/s)", fontsize=12)
        ax.legend(fontsize=11)
        ax.grid(axis="y", alpha=0.4)

    else:  # Boxplot — Ausreißer (Flier) ausgeblendet, sonst überladen der Plot
        n_models = len(models)
        fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5), sharey=True)
        if n_models == 1:
            axes = [axes]
        for ax, mdl in zip(axes, models):
            dm = df[df["model"] == mdl].copy()
            dm["abs_err"] = (dm["pred"] - dm["gt"]).abs()
            order = _WS_LABELS if x_col == "ws_bin" else sorted(dm[x_col].unique())
            groups = [dm[dm[x_col] == v]["abs_err"].dropna().values for v in order]
            ax.boxplot(groups, labels=[str(v) for v in order],
                       showfliers=False,
                       patch_artist=True,
                       boxprops=dict(facecolor=_MODEL_COLORS.get(mdl, "steelblue"), alpha=0.6))
            ax.set_title(mdl, fontsize=12)
            ax.set_xlabel(xlabel, fontsize=11)
            ax.tick_params(axis="x", rotation=45)
            if ax == axes[0]:
                ax.set_ylabel("|Fehler| (m/s)", fontsize=11)
            ax.grid(axis="y", alpha=0.3)

    fold_text = fold if fold != "Average" else "Ø alle Folds"
    fig.suptitle(
        f"{x_axis} | Split: {split} | Metrik: {metric} | Variante: {variant} | {fold_text}",
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    plt.show(); plt.close()


_style5  = {"description_width": "90px"}
_layout5 = widgets.Layout(width="220px")

w5_xaxis   = widgets.Dropdown(options=list(_X_OPTS.keys()), value="WS-Klasse (m/s)",
                               description="X-Achse:", style=_style5, layout=_layout5)
w5_metric  = widgets.Dropdown(options=["RMSE", "MAE"], value="RMSE",
                               description="Metrik:", style=_style5, layout=_layout5)
w5_variant = widgets.Dropdown(options=["All", "BASE", "GRID", "GRID+HIST"], value="BASE",
                               description="Variante:", style=_style5, layout=_layout5)
w5_split, w5_fold = make_split_fold(_style5, _layout5)
w5_type    = widgets.Dropdown(options=["Boxplot", "Linie"], value="Boxplot",
                               description="Plottyp:", style=_style5, layout=_layout5)

out5 = widgets.interactive_output(
    plot_stratified,
    {"x_axis": w5_xaxis, "metric": w5_metric, "variant": w5_variant,
     "fold": w5_fold, "plot_type": w5_type, "split": w5_split}
)
display(widgets.HBox([w5_xaxis, w5_split, w5_metric, w5_variant, w5_fold, w5_type]), out5)

Output()